# 07-units-T — Dimensionally-constrained ozone search WITH temperature in the pool (Bogotá)

Follow-up to `07_o3_units_equation_discovery.ipynb`. That run used the original no-lag selection and
**never offered `temp_2m`**, so it could not say anything about temperature: the earlier wording "the constrained
equation keeps PM2.5 but drops T" was wrong. What stops the free-form lead equation (Eq. 2,
`T/(NO/PM2.5 + 0.634 + w700) + sqrt(NO2*SW/(0.483*RH))`) from passing the unit check is `w700` (Pa/s) being added to a
ratio and a constant with no scale constant of its own.

**One change only** relative to the original constrained run: `temp_2m` (unit `K`, a temperature *difference*, as in the
units audit, notebook A4) is added to the candidate pool. Everything else is identical: the same pool
(`photochemical_potential` decomposed into `no2` and `radiation_mean`), the same units, the same operators, `niterations=200`,
`populations=24`, `maxsize=25`. Symbolic search is stochastic, so it is run for several seeds.

Scoring (same protocol as the other lead equations, notebook A2/A4): leave-one-year-out, constants refitted by non-linear least
squares on the training years, **both** the pooled R2 and the mean of per-year R2, on the same 2919 days as Eq. 2.

In [1]:
# %% Settings (edit here, then Run All)
SEEDS = [0, 1, 2]                      # symbolic search is stochastic: report all of them
PYSR_KW = dict(niterations=200, binary_operators=["+", "-", "*", "/"], unary_operators=["log", "sqrt", "exp"],
               model_selection="best", maxsize=25, populations=24)     # identical to 07_o3_units_equation_discovery
ADD_TEMPERATURE = True                 # the only change relative to the original constrained run
UNIT_MAP = {'no': 'kg/m^3', 'pm25': 'kg/m^3', 'no2': 'kg/m^3', 'rh_2m': '', 'w700': 'Pa/s', 'day_of_week': '',
            'precip_total_mm': 'mm', 'radiation_mean': 'W/m^2',
            'temp_2m': 'K'}            # degC treated as a temperature difference (dimension Theta), as in notebook A4
Y_UNITS = 'kg/m^3'                     # same documented mass-concentration assumption as the PM2.5 notebook
MAX_COMPLEXITY_REPORTED = 25
TAG = "units_temp"                     # output file stem

In [2]:
import sys, json, re, warnings
from pathlib import Path
PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))
import numpy as np, pandas as pd, sympy as sp
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score

DATA_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "bogota"
df = pd.read_parquet(DATA_DIR / "bogota_o3_master_dataset.parquet")
df["date"] = pd.to_datetime(df["date"]); df["year"] = df["date"].dt.year
df = df.dropna(subset=['o3']).reset_index(drop=True)
selection = json.load(open(DATA_DIR / "bogota_o3_feature_selection.json"))
features_orig = selection['selected_no_lag']
features = [f for f in features_orig if f != 'photochemical_potential']
if 'radiation_mean' not in features: features.append('radiation_mean')
if ADD_TEMPERATURE: features.append('temp_2m')
print("pool:", features)
data = df.dropna(subset=features + ['o3']).reset_index(drop=True)
X = data[features].to_numpy(); y = data['o3'].to_numpy()
X_units = [UNIT_MAP[f] for f in features]
print(data.shape, dict(zip(features, X_units)))

pool: ['no', 'pm25', 'rh_2m', 'w700', 'day_of_week', 'no2', 'precip_total_mm', 'radiation_mean', 'temp_2m']
(2919, 220) {'no': 'kg/m^3', 'pm25': 'kg/m^3', 'rh_2m': '', 'w700': 'Pa/s', 'day_of_week': '', 'no2': 'kg/m^3', 'precip_total_mm': 'mm', 'radiation_mean': 'W/m^2', 'temp_2m': 'K'}


In [3]:
from pysr import PySRRegressor
fronts = {}
for seed in SEEDS:
    sr = PySRRegressor(**PYSR_KW, random_state=seed)
    sr.fit(X, y, variable_names=features, X_units=X_units, y_units=Y_UNITS)
    f = sr.equations_[['complexity', 'loss', 'equation']].copy(); f['seed'] = seed
    f.to_csv(DATA_DIR / f"bogota_o3_pysr_{TAG}_pareto_front_seed{seed}.csv", index=False)
    fronts[seed] = f
    print("seed", seed, "done; best:", sr.get_best()["equation"])

Compiling Julia backend...


/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(


┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 1.950e+04
Progress: 45 / 200 total iterations (22.854%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.892e+01  0.000e+00  y[m⁻³ kg] = 20.147[?]
3           4.139e+01  8.356e-02  y[m⁻³ kg] = radiation_mean[m⁻² W] * 0.10532[?]
4           3.868e+01  6.759e-02  y[m⁻³ kg] = sqrt(pm25[m⁻³ kg] * 22.601[?])
5           3.708e+01  4.228e-02  y[m��³ kg] = (pm25[m⁻³ kg] / 2.3429[?]) + 12.451[?]
6           3.444e+01  7.391e-02  y[m⁻³ kg] = sqrt(radiation_mean[m⁻² W] / (8.4225[?...
                                              ] / pm25[m⁻³ kg]))
7           2.956e+01  1.528e-01  y[m⁻³ kg] = sqrt(107.05[?] / (sqrt(no[m⁻³ kg]) / p...
                                              m25[m⁻³ kg]))
8           2.956e+01  -0.000e+00  y[m⁻³ kg] = sqrt(107.05[?]) /

seed 0 done; best: sqrt(radiation_mean / sqrt(no / (no2 * (0.25719875 * pm25))))


/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(
┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 2.110e+04
Progress: 46 / 200 total iterations (23.229%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.892e+01  0.000e+00  y[m⁻³ kg] = 20.148[?]
3           4.139e+01  8.356e-02  y[m⁻³ kg] = radiation_mean[m⁻² W] * 0.10535[?]
4           3.869e+01  6.752e-02  y[m⁻³ kg] = sqrt(pm25[m⁻³ kg] * 22.788[?])
5           3.708e+01  4.245e-02  y[m��³ kg] = (pm25[m⁻³ kg] * 0.42959[?]) + 12.136[?...
                                              ]
6           3.444e+01  7.373e-02  y[m⁻³ kg] = sqrt((radiation_mean[m⁻² W] * pm25[m⁻³...
                                               kg]) * 0.11923[?])
7           2.903e+01  1.709e-01  y[m⁻³ kg] = sqrt((pm25[m⁻³ kg] * 99.529[?]) / sqrt...
                                              (no[m⁻³ kg]))
9           

/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(
┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 1.950e+04
Progress: 43 / 200 total iterations (21.750%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.892e+01  0.000e+00  y[m⁻³ kg] = 20.147[?]
3           4.141e+01  8.329e-02  y[m⁻³ kg] = radiation_mean[m⁻² W] * 0.10443[?]
4           3.868e+01  6.815e-02  y[m⁻³ kg] = sqrt(pm25[m⁻³ kg] / 0.044223[?])
5           3.706e+01  4.284e-02  y[m⁻³ kg] = (pm25[m⁻³ kg] * 0.42401[?]) + 12.357[?...
                                              ]
6           3.647e+01  1.607e-02  y[m⁻³ kg] = sqrt(temp_2m[K] * pm25[m⁻³ kg]) * 1.24...
                                              05[?]
7           3.216e+01  1.258e-01  y[m⁻³ kg] = (radiation_mean[m⁻² W] * (pm25[m⁻³ kg]...
                                               * 0.0020121[?])) + 12.936[?]
9        

## Score every equation on every front: leave-one-year-out, constants refitted

Each numeric constant in the PySR expression becomes a free parameter, started from PySR's value. Evaluated on the same 2919 days
as Eq. 2 (rows with all of `temp_2m, no, pm25, w700, photochemical_potential, rh_2m` and `o3`).

In [4]:
FLOAT = re.compile(r'(?<![A-Za-z_0-9.])\d+\.\d+(?:[eE][+-]?\d+)?')
def make_model(eq, feats):
    vals = []
    def sub(m): vals.append(float(m.group(0))); return f"c{len(vals)-1}"
    s = FLOAT.sub(sub, eq)
    syms = {f: sp.Symbol(f) for f in feats}; cs = [sp.Symbol(f"c{i}") for i in range(len(vals))]
    expr = sp.sympify(s, locals={**syms, **{str(c): c for c in cs}, "sqrt": sp.sqrt, "log": sp.log, "exp": sp.exp})
    used = [f for f in feats if syms[f] in expr.free_symbols]
    keep = [i for i, c in enumerate(cs) if c in expr.free_symbols]
    fn = sp.lambdify([syms[f] for f in used] + [cs[i] for i in keep], expr, "numpy")
    p0 = np.array([vals[i] for i in keep])
    return (lambda Xt, *p: fn(*Xt, *p)), used, p0

O3_FEATS = ["temp_2m", "no", "pm25", "w700", "photochemical_potential", "rh_2m"]
canon = df.dropna(subset=O3_FEATS + ['o3']).dropna(subset=features).reset_index(drop=True)
print("evaluation sample:", len(canon), "days (Eq. 2 uses 2919)")
yy = canon['o3'].to_numpy(float); years = canon['year'].to_numpy()

def score(eq):
    model, used, p0 = make_model(eq, features)
    if not used:                                   # constant equation (complexity 1): no variable, nothing to refit
        const = float(p0[0]) if len(p0) else float(eq)
        pred = np.full(len(yy), const)
        per = [r2_score(yy[years == yr], np.full((years == yr).sum(), np.mean(yy[years != yr]))) for yr in sorted(set(years))]
        oof = np.concatenate([np.full((years == yr).sum(), np.mean(yy[years != yr])) for yr in sorted(set(years))])
        yo = np.concatenate([yy[years == yr] for yr in sorted(set(years))])
        return dict(r2_insample=r2_score(yy, pred), r2_pooled=r2_score(yo, oof), r2_mean_years=float(np.mean(per)), vars="")
    Xt = tuple(canon[f].to_numpy(float) for f in used)
    with np.errstate(all='ignore'), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try: popt, _ = curve_fit(model, Xt, yy, p0=p0, maxfev=20000)
        except Exception: popt = p0
        pred = np.full(len(yy), np.nan)
        for yr in sorted(set(years)):
            te = years == yr
            try: pf, _ = curve_fit(model, tuple(a[~te] for a in Xt), yy[~te], p0=popt, maxfev=20000)
            except Exception: pf = popt
            pred[te] = model(tuple(a[te] for a in Xt), *pf)
        ok = np.all(np.isfinite(pred))
        ins = model(Xt, *popt)
        per = [r2_score(yy[years == yr], pred[years == yr]) for yr in sorted(set(years))] if ok else [np.nan]
        return dict(r2_insample=r2_score(yy, ins) if np.all(np.isfinite(ins)) else np.nan,
                    r2_pooled=r2_score(yy, pred) if ok else np.nan, r2_mean_years=float(np.mean(per)), vars=",".join(used))

rows = []
for seed, f in fronts.items():
    for _, r in f.iterrows():
        if r.complexity > MAX_COMPLEXITY_REPORTED: continue
        rows.append(dict(seed=seed, complexity=int(r.complexity), equation=r.equation, **score(r.equation)))
scored = pd.DataFrame(rows)
scored.to_csv(DATA_DIR / f"bogota_o3_pysr_{TAG}_scored.csv", index=False)
pd.set_option("display.width", 250, "display.max_colwidth", 120)
print(scored[['seed','complexity','r2_pooled','r2_mean_years','r2_insample','vars']].round(3).to_string(index=False))

evaluation sample: 2919 days (Eq. 2 uses 2919)


 seed  complexity  r2_pooled  r2_mean_years  r2_insample                                     vars
    0           1     -0.028         -0.149       -0.000                                         
    0           3      0.124          0.002        0.154                           radiation_mean
    0           4      0.154          0.013        0.209                                     pm25
    0           5      0.185          0.053        0.242                                     pm25
    0           6      0.243          0.113        0.296                      pm25,radiation_mean
    0           7      0.362          0.245        0.407                                  no,pm25
    0           9      0.446          0.349        0.478                   no,pm25,radiation_mean
    0          10      0.463          0.358        0.510                   no,pm25,radiation_mean
    0          11      0.563          0.491        0.580               no,pm25,no2,radiation_mean
    0          13   

## Reading the result
Compare, on the same days: free-form lead equation Eq. 2 (pooled 0.610, mean of years 0.543), the original constrained equation
without temperature (pooled 0.532, mean of years 0.448), and the best constrained equation here at the same complexity (16) and
overall. Look at whether `temp_2m` enters, with which partners, and whether `w700` is still needed.

In [5]:
summ = []
for seed in SEEDS:
    s = scored[scored.seed == seed]
    s16 = s[s.complexity <= 16].sort_values('r2_pooled', ascending=False).head(1)
    sb = s.sort_values('r2_pooled', ascending=False).head(1)
    for tag, d in (("best at complexity <=16", s16), ("best overall", sb)):
        if len(d): summ.append(dict(seed=seed, which=tag, complexity=int(d.complexity.iloc[0]), pooled=d.r2_pooled.iloc[0],
                                    mean_years=d.r2_mean_years.iloc[0], equation=d.equation.iloc[0]))
summary = pd.DataFrame(summ); summary.to_csv(DATA_DIR / f"bogota_o3_pysr_{TAG}_summary.csv", index=False)
print(summary.round(3).to_string(index=False))

 seed                   which  complexity  pooled  mean_years                                                                                                                                                            equation
    0 best at complexity <=16          15   0.577       0.506                                                                                sqrt((radiation_mean / sqrt(rh_2m / (pm25 / ((no / no2) / 18.173006)))) - 22.472061)
    0            best overall          21   0.586       0.514                                        sqrt((((radiation_mean * 1.7886186) + 158.06035) / sqrt(rh_2m / (no2 / ((no / pm25) / (temp_2m * 0.2412534))))) + -90.89786)
    1 best at complexity <=16          15   0.576       0.505                                                                             sqrt(sqrt(no2 / no) * (radiation_mean * (0.070238136 * (pm25 + (0.756424 * temp_2m)))))
    1            best overall          25   0.582       0.511 sqrt(sqrt(no2 / no) * ((0.05297018